# Deep YOLO Transfer — Mechatronics Vision 2026

Evolution of `cnn_transfer.ipynb`. Same training loop, same resume helpers,
but:

- **Bigger base model**: starts from `yolo11m-seg.pt` (~20M params, 8× the
  nano). Same YOLO head, same COCO pretraining, more capacity in the backbone.
- **More epochs**: 300 by default, with `patience=50` so Ultralytics
  early-stops if val mAP plateaus.
- **Feature visualization**: `visualize_features()` in §7 — forward-hooks
  the trained backbone at selected stages and produces (a) spatial attention
  heatmaps per stage and (b) PCA scatter of pooled feature vectors colored
  by class. Paper-ready.

Runs land under `runs_cnn_transfer/deep_ft/` so your `v1.1` run stays
untouched and can be A/B-compared against the new weights.


In [ ]:
import os, sys, json, math, random, shutil, copy
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import yaml
import cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
import matplotlib.pyplot as plt

DEVICE = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
print('device:', DEVICE, '| torch', torch.__version__)

## 0. Configuration (shared)

Imported from `vision_config.py` — the same file `augmentation_pipeline.ipynb`
and `deploy.py` read. Flip `LABEL_KIND` in that file to toggle between
polygon-segmentation training and bounding-box detection training; every
path and model choice below follows automatically.

**Note on `yolo_dataset/`.** Section A.1 builds an Ultralytics-style
`images/{train,val}` + `labels/{train,val}` folder there **using symlinks**
back into `data/`. Your real images and labels stay in `data/` —
`yolo_dataset/` is a rebuildable artifact that Ultralytics consumes.

In [ ]:
import sys
sys.path.insert(0, '/Users/jnoael/Mechatronics_Vision_2026')
from vision_config import *   # all shared settings

# Auto-resolved from LABEL_KIND. Override in this cell for one-off experiments.
LABELS_DIR = resolved_labels_dir()
FT_MODEL   = resolved_yolo_model()

RUNS_DIR.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
summary()
print(f'  active labels dir: {LABELS_DIR}')
print(f'  fine-tune base   : {FT_MODEL}')
print(f'  runs dir         : {RUNS_DIR}')

# --- Deep-transfer overrides (local to this notebook) -----------------
# yolo11m-seg = medium-size YOLO11, pretrained on COCO instance-seg.
# Falls back to the non-seg variant if LABEL_KIND is 'bbox'.
FT_MODEL      = 'yolo11m-seg.pt' if LABEL_KIND == 'polygon' else 'yolo11m.pt'
FT_EPOCHS     = 300            # ceiling; patience early-stops before this if flat
FT_PATIENCE   = 50             # stop if val mAP doesn't improve for N epochs
DEEP_RUN_NAME = 'deep_ft'      # run folder under RUNS_DIR; separate from 'ft' / 'v1.1'

print(f'  deep base model : {FT_MODEL}')
print(f'  deep epochs     : {FT_EPOCHS}  (patience={FT_PATIENCE})')
print(f'  deep run name   : {DEEP_RUN_NAME}')


## 0b. Re-import key helpers from the augmentation notebook

Self-contained copies so this notebook runs standalone. Keep in sync with
`augmentation_pipeline.ipynb` (Sections 2–5).

In [ ]:
def detect_format(tokens):
    n = len(tokens)
    if n == 5: return 'bbox'
    if n >= 7 and (n - 1) % 2 == 0: return 'polygon'
    raise ValueError(f'unrecognized label shape: {n} tokens')

def read_label(path):
    items = []
    text = Path(path).read_text().strip()
    if not text: return items
    for line in text.splitlines():
        parts = line.split()
        if not parts: continue
        cls  = int(parts[0])
        nums = [float(x) for x in parts[1:]]
        kind = detect_format([cls] + nums)
        if kind == 'bbox':
            items.append(dict(cls=cls, kind='bbox',
                              coords=np.array(nums, np.float32)))
        else:
            pts = np.array(nums, np.float32).reshape(-1, 2)
            items.append(dict(cls=cls, kind='polygon', coords=pts))
    return items

def poly_to_bbox(pts):
    x_min, y_min = pts.min(axis=0); x_max, y_max = pts.max(axis=0)
    return np.array([(x_min+x_max)/2, (y_min+y_max)/2,
                     x_max-x_min,    y_max-y_min], np.float32)

def polys_to_mask_stack(polys, h, w):
    stack = np.zeros((len(polys), h, w), np.uint8)
    for i, poly in enumerate(polys):
        pix = (poly * np.array([w, h])).round().astype(np.int32)
        cv2.fillPoly(stack[i], [pix], 1)
    return stack

def crop_instance(img, poly, pad_frac=0.10, out_size=224):
    h, w = img.shape[:2]
    x_min, y_min = poly.min(axis=0); x_max, y_max = poly.max(axis=0)
    px = (x_max - x_min) * pad_frac; py = (y_max - y_min) * pad_frac
    x0 = int(max(0, (x_min - px) * w)); y0 = int(max(0, (y_min - py) * h))
    x1 = int(min(w, (x_max + px) * w)); y1 = int(min(h, (y_max + py) * h))
    if x1 <= x0 or y1 <= y0: return None
    return cv2.resize(img[y0:y1, x0:x1], (out_size, out_size))

def build_aug(img_size=IMAGE_SIZE, train=True):
    """Condensed aug pipeline driven by shared AUG_PROBS from vision_config.
    Full pipeline lives in augmentation_pipeline.ipynb."""
    p = AUG_PROBS
    geo = []
    if train:
        geo = [
            A.HorizontalFlip(p=p['h_flip']),
            A.VerticalFlip(p=p['v_flip']),
            A.RandomRotate90(p=p['rotate_90']),
            A.Affine(rotate=(-ROTATE_DEG, ROTATE_DEG),
                     shear={'x': (-SKEW_DEG, SKEW_DEG),
                            'y': (-SKEW_DEG, SKEW_DEG)},
                     p=max(p['rotate_small'], p['affine_skew'])),
            A.RandomResizedCrop(size=(img_size, img_size),
                                scale=CROP_SCALE, ratio=(0.85, 1.17),
                                p=p['random_crop']),
        ]
    geo.append(A.Resize(img_size, img_size))
    photo = []
    if train:
        photo = [
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3, hue=0.1,
                          p=p['color_jitter']),
            A.ChannelDropout(p=p['channel_dropout']),
            A.CoarseDropout(num_holes_range=CUTOUT_HOLES,
                            hole_height_range=CUTOUT_FRAC,
                            hole_width_range=CUTOUT_FRAC,
                            fill=0, p=p['cutout']),
            A.GaussNoise(std_range=(0.04, 0.12), p=p['gauss_noise']),
            A.GaussianBlur(blur_limit=(3, 7), p=p['gauss_blur']),
        ]
    return A.Compose(geo + photo)


## A. YOLO11 Fine-Tune from COCO (primary path)

### A.1 Train/val split

Classes are identified by filename prefix. We split each class 80/20 so both
folds see every class.

In [ ]:
def build_split(images_dir: Path, labels_dir: Path,
                 out_dir: Path, val_fraction: float = VAL_FRACTION,
                 seed: int = SEED) -> Path:
    """Materialize an Ultralytics-style dataset folder with symlinks and a
    YOLO data.yaml. Stratifies by filename prefix."""
    rng = random.Random(seed)
    by_prefix = {}
    for img in sorted(images_dir.iterdir()):
        if img.suffix.lower() not in {'.png', '.jpg', '.jpeg'}: continue
        prefix = img.stem.split('_')[0]
        by_prefix.setdefault(prefix, []).append(img)

    out_dir.mkdir(parents=True, exist_ok=True)
    for split in ('train', 'val'):
        (out_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (out_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    for prefix, imgs in by_prefix.items():
        rng.shuffle(imgs)
        k = max(1, int(round(len(imgs) * val_fraction)))
        val_set = set(imgs[:k]); train_set = set(imgs[k:])
        for img in imgs:
            split = 'val' if img in val_set else 'train'
            lab = labels_dir / (img.stem + '.txt')
            if not lab.exists(): continue
            dst_i = out_dir / 'images' / split / img.name
            dst_l = out_dir / 'labels' / split / lab.name
            if dst_i.exists(): dst_i.unlink()
            if dst_l.exists(): dst_l.unlink()
            dst_i.symlink_to(img); dst_l.symlink_to(lab)

    yaml_path = out_dir / 'data.yaml'
    yaml_path.write_text(yaml.safe_dump({
        'path':  str(out_dir),
        'train': 'images/train',
        'val':   'images/val',
        'names': CLASS_NAMES,
    }, sort_keys=False))
    print('wrote', yaml_path)
    return yaml_path

# yolo_dataset/ is a symlink-only build folder. Real data stays in data/.
DATA_YAML = build_split(IMAGES_DIR, LABELS_DIR, YOLO_DATASET_DIR)
print(Path(DATA_YAML).read_text())


### A.1.5  Hook custom albumentations into Ultralytics' dataloader

Monkey-patches `ultralytics.data.augment.Albumentations.__init__` so every
training sample passes through our custom photometric pipeline — color
jitter + hue rotate + color gradient + channel dropout + cutout + gauss
noise/blur + Sobel — drawn fresh per sample, every epoch. Probabilities
come from `AUG_PROBS` in `vision_config.py`.

**Geometric** augmentations (flip, rotate, scale, shear, mosaic, mixup,
copy_paste, erasing) stay with Ultralytics — kwargs already set in
`finetune_yolo` (cell below). This hook touches only the photometric layer.

The cell ends with a sanity check that runs a dummy image through the
patched pipeline — if the patch is broken, you'll know here, not after
4 hours of training.

Run this once per kernel, before §A.2. The patch persists for the rest
of the session.


In [ ]:
import albumentations as A
from ultralytics.data.augment import Albumentations as _UltralyticsAlbumentations

# Keep a handle on the original __init__ so we can restore on failure.
_orig_alb_init = _UltralyticsAlbumentations.__init__


# ── Custom image-only transforms (mirror augmentation_pipeline.ipynb §4) ──
class SobelLambda(A.ImageOnlyTransform):
    def __init__(self, p=0.5):
        super().__init__(p=p)
    def apply(self, img, **params):
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        mag = np.sqrt(gx**2 + gy**2)
        hi = np.percentile(mag, 99) + 1e-6
        mag = np.clip(255 * mag / hi, 0, 255).astype(np.uint8)
        edge_rgb = cv2.cvtColor(mag, cv2.COLOR_GRAY2RGB)
        return cv2.addWeighted(img, 0.35, edge_rgb, 0.65, 0)
    def get_transform_init_args_names(self):
        return ()


class RandomColorGradient(A.ImageOnlyTransform):
    def __init__(self, control_points=4, max_shift=30, p=0.5):
        super().__init__(p=p)
        self.control_points = control_points
        self.max_shift      = max_shift
    def apply(self, img, **params):
        h, w = img.shape[:2]
        cp = self.control_points
        grid = np.random.uniform(-self.max_shift, self.max_shift,
                                 size=(cp, cp, 3)).astype(np.float32)
        tint = cv2.resize(grid, (w, h), interpolation=cv2.INTER_CUBIC)
        out  = img.astype(np.float32) + tint
        return np.clip(out, 0, 255).astype(np.uint8)
    def get_transform_init_args_names(self):
        return ('control_points', 'max_shift')


def _build_custom_photometric() -> A.Compose:
    """All image-only (pixel-level) — safe to swap into Ultralytics' image-
    only augmentation slot without touching bboxes or masks."""
    p = AUG_PROBS
    return A.Compose([
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3, hue=0.1,
                      p=p['color_jitter']),
        A.HueSaturationValue(hue_shift_limit=HUE_ROTATE_LIMIT,
                             sat_shift_limit=0, val_shift_limit=0,
                             p=p['hue_rotate']),
        RandomColorGradient(control_points=COLOR_GRADIENT_GRID,
                            max_shift=COLOR_GRADIENT_MAX_SHIFT,
                            p=p['color_gradient']),
        A.ChannelDropout(channel_drop_range=(1, 1), fill=0,
                         p=p['channel_dropout']),
        A.CoarseDropout(num_holes_range=CUTOUT_HOLES,
                        hole_height_range=CUTOUT_FRAC,
                        hole_width_range=CUTOUT_FRAC,
                        fill=0, p=p['cutout']),
        A.GaussNoise(std_range=(0.04, 0.12), p=p['gauss_noise']),
        A.GaussianBlur(blur_limit=(3, 7), p=p['gauss_blur']),
        SobelLambda(p=p['sobel']),
    ])  # no bbox_params: pipeline runs via self.transform(image=im) only


def _patched_alb_init(self, *args, **kwargs):
    """Replace Ultralytics' default (Blur/MedianBlur/ToGray/CLAHE at p=0.01)
    with AUG_PROBS-driven photometric pipeline. Pixel-only, so masks +
    bboxes pass through untouched and Ultralytics' geometric pipeline keeps
    operating normally above this layer."""
    self.p = kwargs.get('p', args[0] if args else 1.0)
    self.transform        = _build_custom_photometric()
    self.contains_spatial = False


# Apply patch
_UltralyticsAlbumentations.__init__ = _patched_alb_init

# --- Sanity check ---
# Mirrors Ultralytics' image-only __call__ path:  self.transform(image=im)['image']
try:
    _test = _UltralyticsAlbumentations()
    assert _test.transform is not None,           'transform did not initialise'
    assert _test.contains_spatial is False,       'contains_spatial flag wrong'
    _dummy = np.random.randint(0, 255, size=(640, 640, 3), dtype=np.uint8)
    _out   = _test.transform(image=_dummy)
    assert 'image' in _out and _out['image'].shape == _dummy.shape, 'output shape mismatch'
    print('✓ Ultralytics Albumentations patched + sanity-check passed')
    print(f'  photometric pipeline: {len(_test.transform.transforms)} transforms')
    print(f'  probabilities (from AUG_PROBS):')
    for k in ('color_jitter', 'hue_rotate', 'color_gradient',
              'channel_dropout', 'cutout', 'gauss_noise',
              'gauss_blur', 'sobel'):
        print(f'    {k:<16} = {AUG_PROBS[k]}')
    print('\nrun §A.2 (finetune_yolo) whenever you\'re ready — augmentation'
          ' is now applied per sample, every epoch.')
except Exception as e:
    _UltralyticsAlbumentations.__init__ = _orig_alb_init
    raise RuntimeError(f'patch verification failed, reverted to Ultralytics default: {e}')


### A.2 Fine-tune

Ultralytics `YOLO.train` has built-in augmentation (mosaic, mixup, hsv,
flips). We enable those and let our external `augmentation_pipeline.ipynb`
cover the additional modes (Sobel, channel dropout, etc.) via the
optional bulk export. If you want *only* our pipeline, export augmented
copies with `bulk_export(...)` in the aug notebook and point `DATA_YAML`
at that folder.

In [ ]:
def finetune_yolo(data_yaml: Path, model: str = FT_MODEL,
                   epochs: int = FT_EPOCHS, batch: int = FT_BATCH,
                   imgsz: int = IMAGE_SIZE, project: Path = RUNS_DIR):
    from ultralytics import YOLO
    m = YOLO(model)
    results = m.train(
        data=str(data_yaml), epochs=epochs, batch=batch, imgsz=imgsz,
        project=str(project), name=DEEP_RUN_NAME, exist_ok=True,
        device=0 if DEVICE == 'cuda' else DEVICE,
        patience=FT_PATIENCE,
        # Ultralytics built-in aug (tuned up)
        hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
        degrees=15.0, translate=0.1, scale=0.5, shear=5.0, perspective=0.0,
        flipud=0.1, fliplr=0.5,
        mosaic=1.0, mixup=0.1, copy_paste=0.1,
        erasing=0.2,
    )
    return m, results

# ▶ Actually train. Weights land in runs_cnn_transfer/ft/weights/best.pt
model, results = finetune_yolo(DATA_YAML)
print('best weights:', model.trainer.best)


### A.3 Evaluate

Produces mAP, per-class metrics, and writes confusion matrix + PR curves to
`runs_cnn_transfer/ft/`.

In [ ]:
def evaluate_yolo(weights_path: Path, data_yaml: Path, imgsz: int = IMAGE_SIZE):
    from ultralytics import YOLO
    m = YOLO(str(weights_path))
    metrics = m.val(data=str(data_yaml), imgsz=imgsz, plots=True,
                    device=0 if DEVICE == 'cuda' else DEVICE)
    print('mAP50:',    metrics.box.map50)
    print('mAP50-95:', metrics.box.map)
    return metrics

best_weights = RUNS_DIR / DEEP_RUN_NAME / 'weights' / 'best.pt'
if best_weights.exists():
    _ = evaluate_yolo(best_weights, DATA_YAML)
else:
    print('no weights yet — run the training cell above first')

### A.4 Resume / Further training

Two small helpers for the two common post-run paths:

- **`resume_training`** — a run crashed mid-epoch. Ultralytics saves
  `last.pt` + `args.yaml` after each epoch in the run folder; this helper
  loads them and picks up at the next epoch with identical config.
- **`continue_training`** — a run finished cleanly and you want to train
  the resulting `best.pt` for N more epochs on new/expanded data (e.g.
  after regenerating Blender renders). Starts a *fresh* run that
  warm-starts from those weights, writes to a different `run_name` so
  the original run stays intact.


In [ ]:
def resume_training(run_name: str = DEEP_RUN_NAME, project: Path = RUNS_DIR):
    """Resume an interrupted run from its last.pt. Reads the run's
    args.yaml automatically and continues at the next epoch."""
    from ultralytics import YOLO
    last_pt = project / run_name / 'weights' / 'last.pt'
    if not last_pt.exists():
        raise FileNotFoundError(
            f'No last.pt at {last_pt}. Either no run exists yet, or it '
            f'finished cleanly (only best.pt remains). Use '
            f'continue_training() to further-train a finished best.pt.'
        )
    print(f'resuming from {last_pt}')
    m = YOLO(str(last_pt))
    results = m.train(resume=True)
    return m, results


def continue_training(weights: Path, data_yaml: Path,
                      epochs: int = 30, batch: int = FT_BATCH,
                      run_name: str = 'deep_ft_continue',
                      project: Path = RUNS_DIR):
    """Warm-start a fresh run from an existing best.pt. Use after
    generating new data to train for a few dozen more epochs without
    touching the original run folder."""
    from ultralytics import YOLO
    weights = Path(weights)
    if not weights.exists():
        raise FileNotFoundError(f'weights not found: {weights}')
    print(f'continuing from {weights} -> {project / run_name}')
    m = YOLO(str(weights))
    results = m.train(
        data=str(data_yaml), epochs=epochs, batch=batch,
        imgsz=IMAGE_SIZE, project=str(project), name=run_name, exist_ok=True,
        device=0 if DEVICE == 'cuda' else DEVICE,
        # Same aug knobs as the initial finetune (cell A.2) for consistency
        hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
        degrees=15.0, translate=0.1, scale=0.5, shear=5.0, perspective=0.0,
        flipud=0.1, fliplr=0.5,
        mosaic=1.0, mixup=0.1, copy_paste=0.1,
        erasing=0.2,
    )
    return m, results

# -- Usage (uncomment when you need it) -----------------------------

# Crash recovery on the default 'ft' run:
# resume_training(run_name=DEEP_RUN_NAME)

# Warm-start fine-tune on freshly regenerated data. Rebuild the split
# first so the new images/labels are picked up:
# DATA_YAML = build_split(IMAGES_DIR, LABELS_DIR, YOLO_DATASET_DIR)
# continue_training(
#     weights   = RUNS_DIR / DEEP_RUN_NAME / 'weights' / 'best.pt',
#     data_yaml = DATA_YAML,
#     epochs    = 35,
#     run_name  = 'deep_ft_continue_v1',
# )


## §7 Feature Visualization

Forward-hook the trained backbone at selected stages, capture activations
for a sample of val images, produce two figures:

1. **Spatial attention heatmaps** — one row per sample, one column per
   hooked stage. Each cell shows the sample image with that stage's
   channel-averaged activation overlaid. Tells you *where* the model
   focuses at progressively deeper layers.
2. **Latent PCA scatter** — one subplot per stage, 2D PCA of the stage's
   globally-pooled feature vectors across sample images, colored by class.
   Classes should become *more separable* with depth — if they don't, the
   backbone isn't learning class-discriminative features at that stage.

Layer indices default to `(4, 6, 8, 10)` — the YOLO11 backbone stages
before the neck. Override `LAYER_INDICES` below if you want different hooks.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

LAYER_INDICES = (4, 6, 8, 10)  # indices into the YOLO backbone's Sequential
N_SAMPLES     = 40              # validation images to probe


def _sample_val(n: int, seed: int = SEED):
    """Return [(img_path, class_id)] for n random samples from LABELS_DIR.
    Class = first int on the first non-empty line of the label file."""
    imgs = [p for p in sorted(IMAGES_DIR.iterdir())
            if p.suffix.lower() in ('.png', '.jpg', '.jpeg')]
    rng = random.Random(seed); rng.shuffle(imgs)
    picked = []
    for p in imgs:
        if len(picked) >= n: break
        lab = LABELS_DIR / (p.stem + '.txt')
        if not lab.exists(): continue
        lines = lab.read_text().strip().splitlines()
        if not lines: continue
        picked.append((p, int(lines[0].split()[0])))
    return picked


def collect_features(weights, samples, layer_indices=LAYER_INDICES):
    """Forward-hook backbone layers, run each sample through YOLO, stack.
    Returns {layer_idx: np.ndarray(N, C, H, W)} and [class_id, ...]."""
    from ultralytics import YOLO
    y = YOLO(str(weights))
    layers = list(y.model.model.children())
    activations = {i: [] for i in layer_indices}
    def mk_hook(i):
        def _hook(m, inp, out):
            activations[i].append(out.detach().cpu().float().numpy())
        return _hook
    handles = [layers[i].register_forward_hook(mk_hook(i)) for i in layer_indices]
    class_ids = []
    try:
        for img_path, cls in samples:
            img = cv2.imread(str(img_path))
            if img is None: continue
            y.predict(img, imgsz=IMAGE_SIZE, verbose=False)
            class_ids.append(cls)
    finally:
        for h in handles: h.remove()
    return ({i: np.concatenate(a, axis=0) for i, a in activations.items() if a},
            class_ids)


def plot_spatial_heatmaps(activations, samples, max_rows=6, save_to=None):
    layer_indices = sorted(activations.keys())
    n_rows = min(len(samples), max_rows)
    n_cols = 1 + len(layer_indices)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2*n_cols, 3.2*n_rows))
    if n_rows == 1: axes = axes[np.newaxis, :]
    for r in range(n_rows):
        img = cv2.cvtColor(cv2.imread(str(samples[r][0])), cv2.COLOR_BGR2RGB)
        axes[r, 0].imshow(img)
        axes[r, 0].set_title('source' if r == 0 else '', fontsize=10)
        axes[r, 0].axis('off')
        for c, idx in enumerate(layer_indices, start=1):
            act = activations[idx][r]
            heat = act.mean(axis=0)
            heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
            heat = cv2.resize(heat, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_CUBIC)
            axes[r, c].imshow(img)
            axes[r, c].imshow(heat, alpha=0.5, cmap='jet')
            axes[r, c].set_title(f'layer {idx}' if r == 0 else '', fontsize=10)
            axes[r, c].axis('off')
    plt.tight_layout()
    if save_to:
        save_to.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_to, dpi=120, bbox_inches='tight')
        print(f'saved {save_to}')
    plt.show()


def plot_latent_pca(activations, class_ids, class_names=CLASS_NAMES, save_to=None):
    layer_indices = sorted(activations.keys())
    n = len(layer_indices)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4.2))
    if n == 1: axes = [axes]
    cls_arr = np.array(class_ids)
    for ax, idx in zip(axes, layer_indices):
        pooled = activations[idx].mean(axis=(2, 3))
        xy = PCA(n_components=2, random_state=0).fit_transform(pooled)
        for cid, cname in class_names.items():
            m = cls_arr == cid
            if m.any():
                ax.scatter(xy[m, 0], xy[m, 1], s=28, alpha=0.75, label=cname)
        ax.set_title(f'layer {idx} — 2D PCA of pooled features')
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
        ax.legend(fontsize=8, loc='best')
    plt.tight_layout()
    if save_to:
        save_to.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_to, dpi=120, bbox_inches='tight')
        print(f'saved {save_to}')
    plt.show()


def visualize_features(weights=None, n_samples=N_SAMPLES,
                       layer_indices=LAYER_INDICES, save_dir=None):
    """End-to-end: sample val images → collect activations → plot both figures."""
    weights = Path(weights or (RUNS_DIR / DEEP_RUN_NAME / 'weights' / 'best.pt'))
    if not weights.exists():
        raise FileNotFoundError(f'weights not found: {weights}. Run finetune first.')
    samples = _sample_val(n_samples)
    print(f'probing {len(samples)} images through {weights.name}...')
    activations, class_ids = collect_features(weights, samples, layer_indices)
    for idx, arr in activations.items():
        print(f'  layer {idx}: {arr.shape}')
    save_heat = save_dir and (Path(save_dir) / 'spatial_heatmaps.png')
    save_pca  = save_dir and (Path(save_dir) / 'latent_pca.png')
    plot_spatial_heatmaps(activations, samples, save_to=save_heat)
    plot_latent_pca(activations, class_ids, save_to=save_pca)
    return activations, class_ids, samples

# -- Usage (uncomment after the finetune completes) -----------------
# activations, class_ids, samples = visualize_features(
#     save_dir = PROJECT_ROOT / 'analysis' / 'feature_viz_deep_ft',
# )


## Usage summary

Typical run:

1. **Rebuild split** — run cells 1–7 (imports, config, helpers, build_split).
2. **Finetune** — run the finetune cell under §A.2. Starts from
   `yolo11m-seg.pt` (COCO pretrained), trains up to 300 epochs on your
   split, writes `runs_cnn_transfer/deep_ft/weights/best.pt`. Patience=50.
3. **Evaluate** — §A.3 cell reads from `deep_ft/weights/best.pt`.
4. **Feature visualization** — §7. Call `visualize_features()` any time
   after step 2 completes. Writes PNGs to the save_dir you pass.

| Situation | Command |
| --- | --- |
| Overnight fresh finetune | `model, results = finetune_yolo(DATA_YAML)` |
| Run died mid-training | `resume_training()` (§A.4) |
| Further-train on new data | `continue_training(weights=..., data_yaml=DATA_YAML, epochs=50, run_name='deep_ft_v2')` |
| Inspect features | `visualize_features(save_dir=PROJECT_ROOT / 'analysis' / 'feature_viz_deep_ft')` |

`cnn_transfer.ipynb` remains the nano baseline for A/B comparison — its
weights live at `runs_cnn_transfer/v1.1/weights/best.pt` and are untouched
by anything this notebook does.
